# MITRE ATT&CK Knowledge Graph — v2.0
**Full pipeline:** Tactics → Techniques → Sub-techniques → Procedures → Groups → Software → CAPEC → CWE → CVE

In [8]:
pip install pandas tqdm requests neo4j


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---
## Cell 1 — Config, driver, constraints

In [2]:
from neo4j import GraphDatabase

URI  = "bolt://localhost:7687"
AUTH = ("neo4j", "kg_mitre_v1.1") 

driver = GraphDatabase.driver(URI, auth=AUTH)

CONSTRAINTS = [
    # Node uniqueness constraints (Essential for optimized MERGE lookups)
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Technique)  REQUIRE t.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (ta:Tactic)    REQUIRE ta.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (g:Group)      REQUIRE g.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Software)   REQUIRE s.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:CAPEC)      REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (w:CWE)        REQUIRE w.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (v:CVE)        REQUIRE v.id IS UNIQUE",
    
    # Fast lookups for cross-layer structural links
    "CREATE INDEX IF NOT EXISTS FOR (t:Technique) ON (t.mitre_id)",
    "CREATE INDEX IF NOT EXISTS FOR (ta:Tactic)   ON (ta.mitre_id)"
]

with driver.session() as s:
    for stmt in CONSTRAINTS:
        s.run(stmt)

print("✅ Foundational constraints and indexing engine initialized.")

✅ Foundational constraints and indexing engine initialized.


---
## Cell 2 — Ingest ATT&CK STIX bundle
Ingests: **Tactics**, **Techniques**, **Sub-techniques**, **Procedures**, **Groups**, **Software**  
Relationships: `BELONGS_TO`, `EXECUTED`, `HAS_SUBTECHNIQUE`, `TARGETS`, `USES`

## Graph schema
```
#MITRE Graph
(g:Group)-[:USES]->(s:Software)-[:EXECUTED]->(p:Procedure)-[:TARGETS]->(t:Technique)-[:BELONGS_TO]->(ta:Tactic)

```

Resource: Locally downloaded enterprise-attack.json file

In [3]:
import json
from stix2 import MemoryStore, Filter

ATTACK_FILE = 'enterprise-attack.json' 

def _get(obj, key, default=None):
    """Unified getter: works for both stix2 objects and plain dicts."""
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

def _id(obj):
    """Get the STIX id field from either a stix2 object or a dict."""
    return obj['id'] if isinstance(obj, dict) else obj.id

def _name(obj):
    return obj['name'] if isinstance(obj, dict) else obj.name

def mitre_id(obj):
    """Extract the MITRE ATT&CK external ID (e.g. T1059, TA0112, G0016)."""
    refs = _get(obj, 'external_references', [])
    for ref in refs:
        src = ref.get('source_name') if isinstance(ref, dict) else getattr(ref, 'source_name', '')
        if src == 'mitre-attack':
            return ref.get('external_id') if isinstance(ref, dict) else getattr(ref, 'external_id', None)
    return _id(obj)  # fallback to STIX id

def url(obj):
    refs = _get(obj, 'external_references', [])
    for ref in refs:
        src = ref.get('source_name') if isinstance(ref, dict) else getattr(ref, 'source_name', '')
        if src == 'mitre-attack':
            return ref.get('url', '') if isinstance(ref, dict) else getattr(ref, 'url', '')
    return ''

def run_full_attack_ingestion(file_path):
    print("Loading STIX bundle...")
    with open(file_path) as f:
        bundle = json.load(f)
    store = MemoryStore(stix_data=bundle)

    with driver.session() as s:
        # ── 1. TACTICS (x-mitre-tactic) ──────────────────────────────────────
        print("Ingesting Tactics...")
        tactics = store.query([Filter('type', '=', 'x-mitre-tactic')])
        for t in tactics:
            mid = mitre_id(t)
            s.run("""
                MERGE (ta:Tactic {stix_id: $stix_id})
                SET ta.mitre_id   = $mid,
                    ta.name       = $name,
                    ta.short_name = $short,
                    ta.description= $desc,
                    ta.url        = $url
            """, stix_id=_id(t), mid=mid, name=_name(t),
                 short=_get(t, 'x_mitre_shortname', ''),
                 desc=_get(t, 'description', ''), url=url(t))
        print(f"  → {len(tactics)} tactics ingested (including TA0112)")

        # ── 2. TECHNIQUES + SUB-TECHNIQUES (attack-pattern) ──────────────────
        print("Ingesting Techniques & Sub-techniques...")
        techniques = store.query([Filter('type', '=', 'attack-pattern')])
        # Filter out revoked/deprecated
        techniques = [t for t in techniques
                      if not t.get('revoked') and not t.get('x_mitre_deprecated')]
        for t in techniques:
            mid = mitre_id(t)
            is_sub = '.' in mid
            platforms = t.get('x_mitre_platforms', [])
            s.run("""
                MERGE (tech:Technique {stix_id: $stix_id})
                SET tech.mitre_id    = $mid,
                    tech.name        = $name,
                    tech.description = $desc,
                    tech.is_subtechnique = $is_sub,
                    tech.platforms   = $platforms,
                    tech.url         = $url
            """, stix_id=t.id, mid=mid, name=t.name,
                 desc=t.get('description', ''), is_sub=is_sub,
                 platforms=platforms, url=url(t))
        print(f"  → {len(techniques)} techniques/sub-techniques")

        # ── 3. GROUPS (intrusion-set) ─────────────────────────────────────────
        print("Ingesting Groups (Threat Actors)...")
        groups = store.query([Filter('type', '=', 'intrusion-set')])
        groups = [g for g in groups
                  if not g.get('revoked') and not g.get('x_mitre_deprecated')]
        for g in groups:
            mid = mitre_id(g)
            aliases = g.get('aliases', [])
            s.run("""
                MERGE (gr:Group {stix_id: $stix_id})
                SET gr.mitre_id   = $mid,
                    gr.name       = $name,
                    gr.aliases    = $aliases,
                    gr.description= $desc,
                    gr.url        = $url
            """, stix_id=g.id, mid=mid, name=g.name,
                 aliases=aliases, desc=g.get('description', ''), url=url(g))
        print(f"  → {len(groups)} groups")

        # ── 4. SOFTWARE (malware + tool) ──────────────────────────────────────
        print("Ingesting Software (Malware + Tools)...")
        software = (
            store.query([Filter('type', '=', 'malware')]) +
            store.query([Filter('type', '=', 'tool')])
        )
        software = [sw for sw in software
                    if not sw.get('revoked') and not sw.get('x_mitre_deprecated')]
        for sw in software:
            mid = mitre_id(sw)
            s.run("""
                MERGE (so:Software {stix_id: $stix_id})
                SET so.mitre_id   = $mid,
                    so.name       = $name,
                    so.type       = $sw_type,
                    so.description= $desc,
                    so.url        = $url
            """, stix_id=sw.id, mid=mid, name=sw.name,
                 sw_type=sw.type, desc=sw.get('description', ''), url=url(sw))
        print(f"  → {len(software)} software items")

        # ── 5. RELATIONSHIPS & PROCEDURES ────────────────────────────────────
        print("Building relationships & isolating structural procedures...")
        rels = store.query([Filter('type', '=', 'relationship')])

        tactic_phase_map = {_get(t, 'x_mitre_shortname'): _id(t) for t in tactics}

        tactic_count = 0
        subtechnique_count = 0
        procedure_count = 0

        # 5a. Technique → Tactic (via kill_chain_phases on each technique)
        for t in techniques:
            for phase in t.get('kill_chain_phases', []):
                short = phase.get('phase_name')
                ta_stix = tactic_phase_map.get(short)
                if ta_stix:
                    s.run("""
                        MATCH (tech:Technique {stix_id: $tech_id})
                        MATCH (ta:Tactic {stix_id: $ta_id})
                        MERGE (tech)-[:BELONGS_TO]->(ta)
                    """, tech_id=t.id, ta_id=ta_stix)
                    tactic_count += 1

        # 5b. Sub-technique → Parent technique
        for rel in rels:
            if rel.relationship_type == 'subtechnique-of':
                s.run("""
                    MATCH (parent:Technique {stix_id: $parent_id})
                    MATCH (sub:Technique   {stix_id: $sub_id})
                    MERGE (parent)-[:HAS_SUBTECHNIQUE]->(sub)
                """, parent_id=rel.target_ref, sub_id=rel.source_ref)
                subtechnique_count += 1

        # 5c. Group/Software USES Technique/Software -> Elevated to Procedure Nodes
        for rel in rels:
            if rel.relationship_type == 'uses':
                proc_desc = _get(rel, 'description', '')
                
                # Derive a clean, distinct identifier string for the intermediate node
                proc_stix_id = f"procedure--{_id(rel).split('--')[-1]}"
                
                s.run("""
                    MATCH (src {stix_id: $src})
                    MATCH (tgt {stix_id: $tgt})
                    
                    // Instantiate the intermediate Procedure context node
                    MERGE (p:Procedure {stix_id: $proc_stix_id})
                    SET p.description = $desc,
                        p.url         = $url
                        
                    // Build structural tracking paths
                    MERGE (src)-[:EXECUTED]->(p)
                    MERGE (p)-[:TARGETS]->(tgt)

                    // NEW: Connect Group directly to Software if src is a Group and tgt is Software
                    WITH src, tgt
                    WHERE src:Group AND tgt:Software
                    MERGE (src)-[:USES]->(tgt)
                """, src=rel.source_ref, tgt=rel.target_ref, 
                     proc_stix_id=proc_stix_id, desc=proc_desc, url=url(rel))
                procedure_count += 1

        print(f"  → {tactic_count} BELONGS_TO (Technique→Tactic)")
        print(f"  → {subtechnique_count} HAS_SUBTECHNIQUE relationships created")
        print(f"  → {procedure_count} Procedure nodes mapped via (:EXECUTED) and (:TARGETS)")

    print("\n✅ ATT&CK Knowledge Graph Pipeline Execution Complete!")

run_full_attack_ingestion(ATTACK_FILE)

Loading STIX bundle...
Ingesting Tactics...
  → 14 tactics ingested (including TA0112)
Ingesting Techniques & Sub-techniques...
  → 691 techniques/sub-techniques
Ingesting Groups (Threat Actors)...
  → 172 groups
Ingesting Software (Malware + Tools)...
  → 784 software items
Building relationships & isolating structural procedures...
  → 887 BELONGS_TO (Technique→Tactic)
  → 477 HAS_SUBTECHNIQUE relationships created
  → 17270 Procedure nodes mapped via (:EXECUTED) and (:TARGETS)

✅ ATT&CK Knowledge Graph Pipeline Execution Complete!


---
## Cell 3 — Get CWE with its chilfOf CWEs and Related_Attack_Pattern(CAPEC)
Ingests: **CWE**, **CAPEC** 

Relationships: `CHILD_OF`, `RELATED_TO`

Resource: http://cwe.mitre.org/data/xml/cwec_latest.xml.zip

In [27]:
import os
import re
import json
import requests
from zipfile import ZipFile
import xml.etree.ElementTree as ET
from neo4j import GraphDatabase

CWE_FILE = "http://cwe.mitre.org/data/xml/cwec_latest.xml.zip"

def download_and_parse_cwe():
    print("[!] Downloading CWE data...")
    response = requests.get(CWE_FILE)
    if response.status_code != 200:
        raise Exception("Failed to download CWE file")
        
    zip_path = "cwec_latest.xml.zip"
    with open(zip_path, 'wb') as f:
        f.write(response.content)
        
    with ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
    os.remove(zip_path)
    
    file_name = re.search(r"cwec_v[\d\.]+\.xml", " ".join(os.listdir())).group()
    print(f"[!] Parsing XML file: {file_name}")
    
    tree = ET.parse(file_name)
    os.remove(file_name)
    return tree.getroot()

def transform_cwe_data(root):
    results = {}
    ns = ""
    m = re.match(r'\{.*\}', root.tag)
    if m: ns = m.group(0)

    weaknesses = root.findall(f".//{ns}Weakness")
    
    for weakness in weaknesses:
        cwe_id = weakness.get("ID")
        name = weakness.get("Name")
        
        # ENRICHMENT: Extract clean textual description
        desc_el = weakness.find(f"{ns}Description")
        description = desc_el.text.strip() if desc_el is not None and desc_el.text else ""
        
        # ENRICHMENT: Extract Likelihood of Exploit
        like_el = weakness.find(f"{ns}Likelihood_Of_Exploit")
        likelihood = like_el.text.strip() if like_el is not None and like_el.text else "Unknown"
        
        # ENRICHMENT: Extract Technical Consequences (Scope & Impact)
        consequences = []
        cons_el = weakness.find(f"{ns}Common_Consequences")
        if cons_el is not None:
            for con in cons_el.findall(f"{ns}Consequence"):
                scopes = [s.text for s in con.findall(f"{ns}Scope") if s.text]
                impacts = [i.text for i in con.findall(f"{ns}Impact") if i.text]
                if scopes:
                    consequences.append(f"[{', '.join(scopes)}] Impact: {', '.join(impacts)}")
        consequences_str = "; ".join(consequences)

        results[cwe_id] = {
            "ID": cwe_id,
            "Name": name,
            "Description": description,
            "Likelihood": likelihood,
            "Consequences": consequences_str,
            "ChildOf": set(),
            "RelatedAttackPatterns": set()
        }
        
        # Interconnection Fix: Parse ALL ChildOf paths across all views
        related_weaknesses = weakness.find(f"{ns}Related_Weaknesses")
        if related_weaknesses is not None:
            for rw in related_weaknesses.findall(f"{ns}Related_Weakness"):
                if rw.get("Nature") == "ChildOf":
                    parent_id = rw.get("CWE_ID")
                    if parent_id:
                        results[cwe_id]["ChildOf"].add(parent_id)
                        
        related_patterns = weakness.find(f"{ns}Related_Attack_Patterns")
        if related_patterns is not None:
            for rap in related_patterns.findall(f"{ns}Related_Attack_Pattern"):
                capec_id = rap.get("CAPEC_ID")
                if capec_id:
                    results[cwe_id]["RelatedAttackPatterns"].add(capec_id.strip())
                    
        results[cwe_id]["ChildOf"] = list(results[cwe_id]["ChildOf"])
        results[cwe_id]["RelatedAttackPatterns"] = list(results[cwe_id]["RelatedAttackPatterns"])
                    
    return results

def ingest_to_neo4j(cwe_dict):
    print("[!] Ingesting enriched CWE data into Neo4j...")
    
    cwe_list = [{
        "id": k, 
        "name": v["Name"], 
        "desc": v["Description"],
        "likelihood": v["Likelihood"],
        "consequences": v["Consequences"],
        "ChildOf": v["ChildOf"], 
        "RelatedAttackPatterns": v["RelatedAttackPatterns"]
    } for k, v in cwe_dict.items()]
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        driver.verify_connectivity()
        
        driver.execute_query("CREATE CONSTRAINT cwe_id_unique IF NOT EXISTS FOR (c:CWE) REQUIRE c.id IS UNIQUE")
        driver.execute_query("CREATE CONSTRAINT capec_id_unique IF NOT EXISTS FOR (a:CAPEC) REQUIRE a.id IS UNIQUE")
        
        print(" -> Generating Enriched CWE Nodes...")
        cwe_query = """
        UNWIND $batch AS item 
        MERGE (c:CWE {id: item.id}) 
        SET c.name = item.name,
            c.description = item.desc,
            c.likelihood_of_exploit = item.likelihood,
            c.common_consequences = item.consequences
        """
        driver.execute_query(cwe_query, batch=cwe_list)
        
        print(" -> Setting structural ChildOf relationships...")
        driver.execute_query("""
            UNWIND $batch AS item
            MATCH (child:CWE {id: item.id})
            UNWIND item.ChildOf AS parentId
            MATCH (parent:CWE {id: parentId})
            MERGE (child)-[:CHILD_OF]->(parent)
        """, batch=cwe_list)
        
        print(" -> Connecting CWE to Related Attack Patterns...")
        driver.execute_query("""
            UNWIND $batch AS item
            MATCH (cwe:CWE {id: item.id})
            UNWIND item.RelatedAttackPatterns AS capecId
            MERGE (capec:CAPEC {id: capecId})
            MERGE (cwe)-[:RELATED_TO]->(capec)
        """, batch=cwe_list)
        
    print("[+] Enriched CWE Ingestion complete.")

if __name__ == "__main__":
    xml_root = download_and_parse_cwe()
    cwe_data = transform_cwe_data(xml_root)
    ingest_to_neo4j(cwe_data)

[!] Downloading CWE data...
[!] Parsing XML file: cwec_v4.20.xml
[!] Ingesting enriched CWE data into Neo4j...
 -> Generating Enriched CWE Nodes...
 -> Setting structural ChildOf relationships...
 -> Connecting CWE to Related Attack Patterns...
[+] Enriched CWE Ingestion complete.


---
## Cell 4 — Ingest CAPEC & Map to MITRE ATT&CK Techniques
Downloads CAPEC catalog 1000.csv, structures it into a local dictionary artifact, and maps it directly onto the existing MITRE ATT&CK core layers.

### Structural Path
`(:CWE) -[:RELATED_TO]-> (:CAPEC) -[:MAPS_TO_ATTACK]-> (:Technique)`

resource = "https://capec.mitre.org/data/csv/1000.csv.zip"


In [28]:
import os
import re
import csv
import json
import requests
from zipfile import ZipFile
from neo4j import GraphDatabase

CAPEC_FILE_URL = "https://capec.mitre.org/data/csv/1000.csv.zip"
CAPEC_FILE = "capec_db.json"

def download_capec():
    print("[!] Downloading CAPEC CSV bundle...")
    response = requests.get(CAPEC_FILE_URL)
    with open("1000.csv.zip", 'wb') as f:
        f.write(response.content)
    with ZipFile("1000.csv.zip", 'r') as zip_ref:
        zip_ref.extractall()
    os.remove("1000.csv.zip")
    
    with open("1000.csv", 'r', encoding='utf-8') as f:
        lines = f.readlines()
    os.remove("1000.csv")
    
    header_idx = 0
    for idx, line in enumerate(lines):
        if "ID,Name,Abstraction" in line or "'ID,Name,Abstraction" in line:
            header_idx = idx
            break
            
    reader = csv.DictReader(lines[header_idx:])
    return [row for row in reader]

def format_capec_to_techniques(capec_list):
    print("[!] Formatting enriched CAPEC entries and structural cross-links...")
    capec_data = {}
    
    for capec in capec_list:
        capec_id = capec.get("'ID") or capec.get("ID")
        if not capec_id:
            continue
            
        name = capec.get("Name", "")
        description = capec.get("Description", "")
        severity = capec.get("Typical Severity", "Unknown")
        likelihood = capec.get("Likelihood Of Attack", "Unknown")
        taxonomy_mappings = capec.get("Taxonomy Mappings", "")
        related_weaknesses_str = capec.get("Related Weaknesses", "")
        
        # 1. Map to ATT&CK Techniques (Tokenized to eliminate false positives)
        techniques = []
        blocks = taxonomy_mappings.split("::")
        for block in blocks:
            if "TAXONOMY NAME:ATTACK" in block:
                match = re.search(r"ENTRY ID:(T?\d+(?:\.\d+)?)", block)
                if match:
                    entry = match.group(1)
                    tech_id = entry if entry.startswith('T') else f"T{entry}"
                    if tech_id not in techniques:
                        techniques.append(tech_id)
                        
        # Interconnection Fix: Cross-reference related CWE IDs listed inside the CAPEC data
        # Handles raw data layouts like "::NATURE:ChildOf:CWE ID:732::" or plain "732" lists
        cwes = []
        cwe_matches = re.findall(r"(?:CWE ID:|\b)(\d+)\b", related_weaknesses_str)
        for cwe_match in cwe_matches:
            if cwe_match not in cwes:
                cwes.append(cwe_match.strip())
                        
        capec_data[capec_id.strip()] = {
            "name": name.strip(),
            "description": description.strip(),
            "severity": severity.strip() if severity else "Unknown",
            "likelihood": likelihood.strip() if likelihood else "Unknown",
            "techniques": techniques,
            "related_cwes": cwes
        }

    with open(CAPEC_FILE, 'w', encoding='utf-8') as f:
        json.dump(capec_data, f, indent=4)
        
    return capec_data

def ingest_capec_links(capec_dict):
    print(f"[!] Processing ingestion for {len(capec_dict)} enriched CAPEC definitions...")
    
    batch_list = [
        {
            "capec_id": k, 
            "name": v["name"], 
            "desc": v["description"],
            "severity": v["severity"],
            "likelihood": v["likelihood"],
            "techniques": v["techniques"],
            "related_cwes": v["related_cwes"]
        } 
        for k, v in capec_dict.items()
    ]
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        driver.verify_connectivity()
        
        driver.execute_query("CREATE CONSTRAINT capec_id_unique IF NOT EXISTS FOR (a:CAPEC) REQUIRE a.id IS UNIQUE")
        
        print(" -> Synchronizing Enriched CAPEC nodes...")
        enrich_query = """
        UNWIND $batch AS item
        MERGE (capec:CAPEC {id: item.capec_id})
        SET capec.name = item.name,
            capec.description = item.desc,
            capec.typical_severity = item.severity,
            capec.likelihood_of_attack = item.likelihood
        """
        driver.execute_query(enrich_query, batch=batch_list)
        
        print(" -> Mapping CAPEC edges to ATT&CK Techniques...")
        link_query = """
        UNWIND $batch AS item
        MATCH (capec:CAPEC {id: item.capec_id})
        UNWIND item.techniques AS techId
        MATCH (tech:Technique {mitre_id: techId})
        MERGE (capec)-[:MAPS_TO_ATTACK]->(tech)
        """
        driver.execute_query(link_query, batch=batch_list)
        
        # Interconnection Layer: Establish back-links from CWE to CAPEC discovered in the CSV data
        print(" -> Creating missing CWE-to-CAPEC back-links...")
        backlink_query = """
        UNWIND $batch AS item
        MATCH (capec:CAPEC {id: item.capec_id})
        UNWIND item.related_cwes AS cweId
        MATCH (cwe:CWE {id: cweId})
        MERGE (cwe)-[:RELATED_TO]->(capec)
        """
        driver.execute_query(backlink_query, batch=batch_list)
        
    print("[+] Enriched CAPEC to Technique and CWE pipeline update complete.")

if __name__ == "__main__":
    raw_list = download_capec()
    formatted_dict = format_capec_to_techniques(raw_list)
    ingest_capec_links(formatted_dict)

[!] Downloading CAPEC CSV bundle...
[!] Formatting enriched CAPEC entries and structural cross-links...
[!] Processing ingestion for 559 enriched CAPEC definitions...
 -> Synchronizing Enriched CAPEC nodes...
 -> Mapping CAPEC edges to ATT&CK Techniques...
 -> Creating missing CWE-to-CAPEC back-links...
[+] Enriched CAPEC to Technique and CWE pipeline update complete.


Cell 5 — Ingest CVE Nodes and Map to CWE Weaknesses
---
Downloads CVEs from NVD site with user-defined start and end years. 

### Structural Path
`(:CVE) -[:EXPLOITS_WEAKNESS]-> (:CWE) -[:RELATED_TO]-> (:CAPEC) -[:MAPS_TO_ATTACK]-> (:Technique)`

resource = "https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-{year}.json.gz"


In [58]:
import os
import re
import json
import gzip
import time
import requests
from re import match
from tqdm import tqdm
from neo4j import GraphDatabase

DOWNLOAD_DIR = "CVEfromNVD"
URI, AUTH = "bolt://localhost:7687", ("neo4j", "kg_mitre_v1.1")

BASE_URL = "https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-{year}.json.gz"
START_YEAR = 2002
END_YEAR   = 2026  # update as needed

# ── 1. Download ────────────────────────────────────────────────────────────────

def download_feeds(download_dir: str = DOWNLOAD_DIR) -> None:
    """Download all annual NVD 2.0 JSON feeds that are not already on disk."""
    os.makedirs(download_dir, exist_ok=True)

    years = range(START_YEAR, END_YEAR + 1)
    print(f"[*] Checking / downloading feeds for {START_YEAR}–{END_YEAR} …")

    for year in years:
        filename  = f"nvdcve-2.0-{year}.json.gz"
        dest_path = os.path.join(download_dir, filename)

        if os.path.exists(dest_path):
            print(f"    [=] {filename} already exists, skipping.")
            continue

        url = BASE_URL.format(year=year)
        print(f"    [↓] Downloading {filename} …", end=" ", flush=True)
        try:
            resp = requests.get(url, stream=True, timeout=60)
            resp.raise_for_status()

            total = int(resp.headers.get("content-length", 0))
            with open(dest_path, "wb") as f, tqdm(
                total=total, unit="B", unit_scale=True,
                desc=filename, leave=False
            ) as bar:
                for chunk in resp.iter_content(chunk_size=65536):
                    f.write(chunk)
                    bar.update(len(chunk))

            print("done.")
        except requests.HTTPError as e:
            print(f"HTTP error {e.response.status_code} — skipping.")
        except Exception as e:
            print(f"Error: {e} — skipping.")

        time.sleep(0.5)   # be polite to NVD servers

# ── 2. Parse & ingest ──────────────────────────────────────────────────────────

def extract_cwes(weaknesses: list) -> list:
    """
    Return a list of numeric CWE id strings (e.g. '79') from a CVE's
    weaknesses array.  Primary entries take precedence over Secondary ones.
    """
    cwe_list: list[str] = []

    def _collect(type_filter: str) -> bool:
        found = False
        for entry in weaknesses:
            if entry.get("type", "") != type_filter:
                continue
            for desc in entry.get("description", []):          # desc is a dict
                value = desc.get("value", "").strip()
                if match(r"CWE-\d{1,4}$", value):
                    num = value.split("-")[1]                   # "79"
                    if num not in cwe_list:
                        cwe_list.append(num)
                        found = True
        return found

    if not _collect("Primary"):
        _collect("Secondary")

    return cwe_list


def parse_and_ingest_nvd_20_feeds(download_dir: str = DOWNLOAD_DIR) -> None:
    gz_files = sorted(
        f for f in os.listdir(download_dir)
        if f.startswith("nvdcve-2.0-") and f.endswith(".json.gz")
    )
    print(f"[!] Found {len(gz_files)} annual JSON 2.0 archive files.")

    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        driver.verify_connectivity()
        driver.execute_query(
            "CREATE CONSTRAINT cve_id_unique IF NOT EXISTS "
            "FOR (v:CVE) REQUIRE v.id IS UNIQUE"
        )

        for file_name in gz_files:
            file_path = os.path.join(download_dir, file_name)
            batch_list: list[dict] = []

            year_match = re.search(r"\d{4}", file_name)
            year_label = year_match.group() if year_match else "Batch"

            print(f"\n[!] Decompressing and reading: {file_name}")
            with gzip.open(file_path, "rt", encoding="utf-8") as f:
                payload = json.load(f)

            for item in tqdm(
                payload.get("vulnerabilities", []),
                desc=f"Processing Year {year_label}",
                unit="CVE",
            ):
                cve_inner = item.get("cve", {})
                cve_id    = cve_inner.get("id", "")
                if not cve_id:
                    continue

                batch_list.append({
                    "cve_id": cve_id,
                    "cwes":   extract_cwes(cve_inner.get("weaknesses", [])),
                })

            # ── Neo4j ingestion in chunks ──────────────────────────────────
            chunk_size = 5000
            for i in range(0, len(batch_list), chunk_size):
                chunk = batch_list[i : i + chunk_size]

                # Step 1 – create/update CVE nodes
                driver.execute_query(
                    "UNWIND $batch AS item "
                    "MERGE (v:CVE {id: item.cve_id})",
                    batch=chunk,
                )

                # Step 2 – link to existing CWE nodes
                driver.execute_query(
                    """
                    UNWIND $batch AS item
                    MATCH (v:CVE {id: item.cve_id})
                    UNWIND item.cwes AS cweId
                    MATCH (w:CWE {id: cweId})
                    MERGE (v)-[:EXPLOITS_WEAKNESS]->(w)
                    """,
                    batch=chunk,
                )

    print("\n[+] All feeds parsed and ingested successfully.")


# ── 3. Entry point ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    download_feeds()
    parse_and_ingest_nvd_20_feeds()

[*] Checking / downloading feeds for 2002–2026 …
    [↓] Downloading nvdcve-2.0-2002.json.gz … 

done.
    [↓] Downloading nvdcve-2.0-2003.json.gz … 

done.
    [↓] Downloading nvdcve-2.0-2004.json.gz … 

done.
    [↓] Downloading nvdcve-2.0-2005.json.gz … 

done.
    [↓] Downloading nvdcve-2.0-2006.json.gz … 

...


[!] Decompressing and reading: nvdcve-2.0-2025.json.gz


Processing Year 2025: 100%|██████████| 44784/44784 [00:01<00:00, 36007.90CVE/s]



[!] Decompressing and reading: nvdcve-2.0-2026.json.gz


Processing Year 2026: 100%|██████████| 27067/27067 [00:01<00:00, 14181.11CVE/s]



[+] All feeds parsed and ingested successfully.
